This module prepares trajecotry data for the analysis by removing trajceotries that do not interect with shallow waters and cleaning the data from duplicated entries and performing feautre engineering.

In [ ]:
import geopandas as gpd

ASSUMED_DRAFT = 1

shallow = gpd.read_file("../data/processed/shallow_waters.gpkg")


In [ ]:
trajcetory_gdf = gpd.read_file("../data/processed/historical_tracks.gpk")

In [ ]:
import pandas as pd

def normalize_timestamps(ts_list):
    if ts_list is None or pd.isna(ts_list):
        return None
    
    if isinstance(ts_list, str):
        try:
            ts_list = eval(ts_list)  # safe here if it's exactly like your example
        except:
            return None

    try:
        # Parse timestamps
        ts = pd.to_datetime(ts_list, utc=True, errors='coerce')
        
        # Remove invalid entries
        ts = ts.dropna()
        if len(ts) == 0:
            return None
        
        ts = ts.round("s")   # removes ms differences like .854 vs .44
        
        ts = ts.sort_values()
        
        return "|".join(ts.strftime("%Y-%m-%d %H:%M:%S"))
    
    except Exception:
        return None

In [ ]:
trajcetory_gdf["ts_norm"] = (
    trajcetory_gdf["timestamps"]
    .apply(normalize_timestamps)
)

In [ ]:
trajcetory_gdf["geom_norm"] = (
    trajcetory_gdf.geometry
    .apply(lambda g: g.wkt if g is not None else None)
)

In [ ]:
subset_cols = ["boatId", "trackDate", "ts_norm", "geom_norm", "directions"]

cleaned = trajcetory_gdf.drop_duplicates(subset=subset_cols)

In [ ]:
cleaned = cleaned.drop(columns=["ts_norm", "geom_norm"])

In [ ]:
if cleaned.crs != shallow.crs:
    cleaned = cleaned.to_crs(shallow.crs)

In [ ]:
cleaned_trajectories_in_shallow_waters = gpd.sjoin(cleaned, shallow, how="inner", predicate="intersects")

In [ ]:
cleaned_trajectories_in_shallow_waters = cleaned_trajectories_in_shallow_waters.loc[~cleaned_trajectories_in_shallow_waters.index.duplicated(keep='first')]

In [ ]:
cleaned_trajectories_in_shallow_waters["Heuristic_Tide_Necessity"] = (
    (ASSUMED_DRAFT - cleaned_trajectories_in_shallow_waters["DRVAL2"]) > 0
).map({True: "Tide Dependent", False: "Tide Independent"})

In [ ]:
cleaned_trajectories_in_shallow_waters["traj_id"] = cleaned_trajectories_in_shallow_waters.index.astype(str)

In [ ]:
cleaned_trajectories_in_shallow_waters.to_file("whole_trajectories_in_shallow_waters_cleaned.gpkg", driver="GPKG")